In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os

torch.manual_seed(42)

# Save everything next to this file
out_dir = os.path.dirname(os.path.abspath(__file__))

# ---------- 1. Load MNIST ----------
transform = transforms.ToTensor()
full_train = datasets.MNIST("data", train=True, download=True, transform=transform)
test_set = datasets.MNIST("data", train=False, download=True, transform=transform)

train_set, val_set = random_split(
    full_train, [50000, 10000], generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)
print("Train:", len(train_set), "| Validation:", len(val_set), "| Test:", len(test_set))

# ---------- 2. Model ----------
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 10),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


def evaluate(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            total_loss += loss_fn(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


# ---------- 3. Train ----------
train_losses, val_losses, val_accs = [], [], []
for epoch in range(10):
    model.train()
    running, n = 0.0, 0
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(y)
        n += len(y)
    train_losses.append(running / n)
    v_loss, v_acc = evaluate(val_loader)
    val_losses.append(v_loss)
    val_accs.append(v_acc)
    print(f"Epoch {epoch + 1:2d}: train loss={train_losses[-1]:.4f}, "
          f"val loss={v_loss:.4f}, val acc={v_acc * 100:.2f}%")

# ---------- 4. Loss curves ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_losses, label="Train")
axes[0].plot(val_losses, label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()
axes[1].plot([a * 100 for a in val_accs], color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation accuracy (%)")
axes[1].set_title("Validation accuracy")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "loss_curves.png"), dpi=120)
plt.close()

# ---------- 5. Test the model ----------
model.eval()
all_pred, all_true, all_conf, all_img = [], [], [], []
with torch.no_grad():
    for x, y in test_loader:
        probs = torch.softmax(model(x), dim=1)
        c, p = probs.max(1)
        all_pred.append(p)
        all_true.append(y)
        all_conf.append(c)
        all_img.append(x)

pred = torch.cat(all_pred)
true = torch.cat(all_true)
conf = torch.cat(all_conf)
imgs = torch.cat(all_img)

test_acc = (pred == true).float().mean().item()
print(f"\nTest accuracy: {test_acc * 100:.2f}%")

# ---------- 6. Confusion matrix ----------
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(true.tolist(), pred.tolist()):
    cm[t, p] += 1

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel("Predicted digit")
plt.ylabel("True digit")
plt.title("Confusion matrix (test set)")
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=120)
plt.close()

# ---------- 7. Precision and recall ----------
print("\nDigit | Precision | Recall")
for d in range(10):
    tp = cm[d, d]
    precision = tp / cm[:, d].sum()
    recall = tp / cm[d, :].sum()
    print(f"  {d}   |   {precision:.3f}   | {recall:.3f}")

# ---------- 8. Sample predictions ----------
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i][0], cmap="gray")
    ok = pred[i] == true[i]
    ax.set_title(f"Pred {pred[i].item()} / True {true[i].item()}",
                 color="green" if ok else "red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "sample_predictions.png"), dpi=120)
plt.close()

# ---------- 9. Wrong predictions ----------
wrong = (pred != true).nonzero().flatten()
print(f"\n{len(wrong)} wrong predictions out of {len(true)}")

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, idx in zip(axes.flat, wrong[:10]):
    ax.imshow(imgs[idx][0], cmap="gray")
    ax.set_title(f"True {true[idx].item()} -> Pred {pred[idx].item()}\n"
                 f"confidence {conf[idx].item() * 100:.0f}%", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "wrong_predictions.png"), dpi=120)
plt.close()

pairs = sorted([(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j],
               reverse=True)
print("\nMost common mistakes:")
for count, t, p in pairs[:5]:
    print(f"  true {t} predicted as {p}: {count} times")

print("\nDone. 4 pictures saved in:", out_dir)

NameError: name '__file__' is not defined